# Qwen2.5-3B SMILES Generator — Stage 2: Specjalizacja EGFR (wersja czysta)

Wszystkie poprawki wbudowane od początku:
- `CUDA_VISIBLE_DEVICES=0` (1 GPU — unika błędu entropii TRL na multi-GPU)
- `ACCELERATE_MIXED_PRECISION=no`, `fp16=False`, `bf16=False` (unika błędu GradScaler/BFloat16)
- `save_steps=50` (częste checkpointy — bezpieczne na wypadek przerwania)
- poprawiony SMARTS dla chinazoliny/pirymidyny (EGFR-like check)
- **brak `os._exit()` w środku** — notebook da się odpalić jako Quick Save / commit od początku do końca

**Setup (Kaggle):** Accelerator = GPU T4 x2, Internet = On, dodaj jako Input dataset z adapterem Stage 1.

## 1. Env vars + instalacja (wszystko w jednej komórce)

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['ACCELERATE_MIXED_PRECISION'] = 'no'

!pip install -q --upgrade 'transformers>=4.45,<4.57' 'tokenizers>=0.20' 'huggingface_hub<1.0'
!pip install -q -U 'accelerate>=1.0' 'peft>=0.13' 'bitsandbytes>=0.44' 'trl>=0.20' 'datasets>=3.0'
!pip install -q rdkit chembl_webresource_client

print('Instalacja OK — kontynuuj bez restartu kernela.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 80.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 8.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 110.8 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 

In [2]:
import torch

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 0: {torch.cuda.get_device_name(0)}')

import transformers, trl, peft
print(f'transformers: {transformers.__version__}')
print(f'trl: {trl.__version__}')
print(f'peft: {peft.__version__}')

CUDA: True
GPU 0: Tesla T4


2026-06-13 10:57:39.680036: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781348259.869230      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781348259.918481      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781348260.357001      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781348260.357049      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781348260.357055      58 computation_placer.cc:177] computation placer alr

transformers: 4.56.2
trl: 1.6.0
peft: 0.19.1


## 2. Pobieranie danych EGFR z ChEMBL

EGFR = `CHEMBL203`. Bierzemy związki z IC50/Ki/Kd < 1000 nM ("aktywne").

In [3]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

EGFR_TARGET_ID = 'CHEMBL203'
ACTIVITY_THRESHOLD_NM = 1000
MAX_COMPOUNDS = 10_000

activity = new_client.activity

print(f'Pobieranie aktywności dla {EGFR_TARGET_ID}...')
results = activity.filter(
    target_chembl_id=EGFR_TARGET_ID,
    standard_type__in=['IC50', 'Ki', 'Kd'],
    standard_units='nM',
    standard_value__lte=ACTIVITY_THRESHOLD_NM,
).only(
    'canonical_smiles', 'standard_value', 'standard_type', 'molecule_chembl_id'
)

rows = []
for i, r in enumerate(results):
    if r.get('canonical_smiles'):
        rows.append(r)
    if len(rows) >= MAX_COMPOUNDS:
        break
    if i % 1000 == 0 and i > 0:
        print(f'  ...przetworzono {i}, zebrano {len(rows)} związków')

df_egfr = pd.DataFrame(rows)
print(f'\nZebrano {len(df_egfr):,} wpisów aktywności EGFR')

df_egfr.to_csv('/kaggle/working/egfr_raw.csv', index=False)
print('Zapisano /kaggle/working/egfr_raw.csv')

Pobieranie aktywności dla CHEMBL203...
  ...przetworzono 1000, zebrano 1001 związków
  ...przetworzono 2000, zebrano 2001 związków
  ...przetworzono 3000, zebrano 3001 związków
  ...przetworzono 4000, zebrano 3998 związków
  ...przetworzono 5000, zebrano 4998 związków
  ...przetworzono 6000, zebrano 5998 związków
  ...przetworzono 7000, zebrano 6998 związków
  ...przetworzono 8000, zebrano 7998 związków
  ...przetworzono 9000, zebrano 8998 związków
  ...przetworzono 10000, zebrano 9998 związków

Zebrano 10,000 wpisów aktywności EGFR
Zapisano /kaggle/working/egfr_raw.csv


## 3. Konfiguracja

1 GPU → `GRAD_ACCUM=8` (efektywny batch = 4 x 8 x 1 = 32, taki sam jak na 2 GPU z GRAD_ACCUM=4).

In [4]:
ADAPTER_PATH_STAGE1 = '/kaggle/input/datasets/agatazajc/adaptery/qwen25_3b_smiles_lora'  # ← EDYTUJ jeśli inna ścieżka

MODEL_NAME = 'Qwen/Qwen2.5-3B'

DATA_SIZE = 10_000
NUM_EPOCHS = 3
LEARNING_RATE = 5e-5
BATCH_SIZE = 4
GRAD_ACCUM = 8        # 1 GPU

PROMPT_PREFIX = 'Generate a drug-like molecule active against EGFR: '

WORK_DIR = '/kaggle/working'
ADAPTER_PATH_STAGE2 = f'{WORK_DIR}/qwen25_3b_smiles_egfr_lora'

print(f'Model bazowy:        {MODEL_NAME}')
print(f'Adapter Stage 1:     {ADAPTER_PATH_STAGE1}')
print(f'Adapter Stage 2 → :  {ADAPTER_PATH_STAGE2}')

Model bazowy:        Qwen/Qwen2.5-3B
Adapter Stage 1:     /kaggle/input/datasets/agatazajc/adaptery/qwen25_3b_smiles_lora
Adapter Stage 2 → :  /kaggle/working/qwen25_3b_smiles_egfr_lora


## 4. Czyszczenie danych SMILES (RDKit)

In [5]:
import random
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

raw = df_egfr['canonical_smiles'].dropna().astype(str).tolist()
print(f'Surowe SMILES: {len(raw):,}')

clean = set()
for smi in raw:
    mol = Chem.MolFromSmiles(smi)
    if mol and 10 <= mol.GetNumHeavyAtoms() <= 60:
        clean.add(Chem.MolToSmiles(mol, canonical=True))

smiles_list = list(clean)
print(f'Unikalne poprawne (10-60 atomów): {len(smiles_list):,}')

if len(smiles_list) > DATA_SIZE:
    random.seed(42)
    smiles_list = random.sample(smiles_list, DATA_SIZE)
    print(f'Próbkowano do {len(smiles_list):,}')

print(f'\nFinalny zbiór treningowy: {len(smiles_list):,} SMILES')

Surowe SMILES: 10,000
Unikalne poprawne (10-60 atomów): 5,785

Finalny zbiór treningowy: 5,785 SMILES


## 5. Model bazowy w 4-bit

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Wczytywanie {MODEL_NAME}...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    dtype=torch.float16,
)
base_model = prepare_model_for_kbit_training(base_model)

# Wymuś fp16 dla wszystkich tensorów bazowego modelu (kompatybilność T4)
for name, p in base_model.named_parameters():
    if p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)
for name, b in base_model.named_buffers():
    if b.dtype == torch.bfloat16:
        b.data = b.data.to(torch.float16)
base_model.config.torch_dtype = torch.float16

print(f'GPU 0: {torch.cuda.memory_allocated(0)/1e9:.2f} GB used')
print('Model bazowy gotowy')

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Wczytywanie Qwen/Qwen2.5-3B...


...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


GPU 0: 2.68 GB used
Model bazowy gotowy


## 6. Wczytanie adaptera Stage 1 (punkt startowy)

In [7]:
print(f'Wczytywanie adaptera Stage 1 z {ADAPTER_PATH_STAGE1}...')
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH_STAGE1,
    is_trainable=True,
)

# Trenowalne wagi LoRA -> fp32; wszystko inne -> fp16 (usuwa resztki bf16)
for name, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)
    elif p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)

for name, b in model.named_buffers():
    if b.dtype == torch.bfloat16:
        b.data = b.data.to(torch.float16)

bf16_remaining = sum(1 for p in model.parameters() if p.dtype == torch.bfloat16)
print(f'Pozostałe tensory bf16: {bf16_remaining}')

model.print_trainable_parameters()
print('Adapter Stage 1 wczytany — gotowy do kontynuacji treningu')

Wczytywanie adaptera Stage 1 z /kaggle/input/datasets/agatazajc/adaptery/qwen25_3b_smiles_lora...
Pozostałe tensory bf16: 0
trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031
Adapter Stage 1 wczytany — gotowy do kontynuacji treningu


## 7. Trening Stage 2

`fp16=False, bf16=False` — unika błędu GradScaler/BFloat16. 1 GPU — unika błędu entropii TRL
na multi-GPU. `save_steps=50` — częste checkpointy.

In [8]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

training_texts = [
    f'{PROMPT_PREFIX}{smi}{tokenizer.eos_token}'
    for smi in smiles_list
]
train_ds = Dataset.from_dict({'text': training_texts})
print(f'Zbiór treningowy: {len(train_ds):,} przykładów')

training_args = SFTConfig(
    output_dir=f'{WORK_DIR}/training_checkpoints_stage2',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_length=256,
    dataset_text_field='text',
    packing=False,
    logging_steps=25,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    optim='paged_adamw_8bit',
    report_to='none',
    ddp_find_unused_parameters=False,
    disable_tqdm=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

print('=== Start treningu Stage 2 ===')
trainer.train()

trainer.save_model(ADAPTER_PATH_STAGE2)
print(f'\nAdapter Stage 2 zapisany w {ADAPTER_PATH_STAGE2}')

Zbiór treningowy: 5,785 przykładów


/tmp/ipykernel_58/2528666519.py:11: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/5785 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5785 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


=== Start treningu Stage 2 ===


Step,Training Loss
25,1.244700
50,0.601500
75,0.524900
100,0.482800
125,0.467500
150,0.442700
175,0.429700
200,0.383100
225,0.365600
250,0.359700



Adapter Stage 2 zapisany w /kaggle/working/qwen25_3b_smiles_egfr_lora


## 8. Test generowania — validity + EGFR-like scaffold (poprawiony SMARTS)

In [15]:
import re

model.eval()

def clean_smiles(text):
    text = text.strip().split('\n')[0].strip()
    match = re.search(r'[A-Za-z\[]', text)
    if match:
        text = text[match.start():]
    return text


def generate_smiles(n=20, temperature=0.7, max_new_tokens=100):
    inputs = tokenizer([PROMPT_PREFIX] * n, return_tensors='pt', padding=True).to(model.device)
    input_length = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    return [clean_smiles(tokenizer.decode(out[input_length:], skip_special_tokens=True)) for out in outputs]


print('Generowanie 20 kandydatów...\n')
samples = generate_smiles(n=20)

# Poprawione wzorce — szkielety charakterystyczne dla inhibitorów EGFR
quinazoline = Chem.MolFromSmarts('c1ccc2ncncc2c1')   # chinazolina (Gefitinib, Erlotinib)
pyrimidine  = Chem.MolFromSmarts('c1ncncc1')         # pirymidyna (Osimertinib i inne)

valid, egfr_like = 0, 0
for i, smi in enumerate(samples, 1):
    mol = Chem.MolFromSmiles(smi)
    status = '+' if mol else '-'
    flag = ''
    if mol:
        valid += 1
        if mol.HasSubstructMatch(quinazoline) or mol.HasSubstructMatch(pyrimidine):
            flag = ' [EGFR-like scaffold]'
            egfr_like += 1
    print(f'{i:2d}. [{status}] {smi}{flag}')

print(f'\nValidity:   {valid}/{len(samples)} ({100*valid/len(samples):.0f}%)')
print(f'EGFR-like:  {egfr_like}/{len(samples)} ({100*egfr_like/len(samples):.0f}%)')

Generowanie 20 kandydatów...

 1. [+] CCCCCCOc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1NC(=O)/C=C/CNC1CC1 [EGFR-like scaffold]
 2. [+] c1ccccc1Cn1ccc2ccc(Nc3ncnn4ccc(CN5CCCCC5)c34)cc21
 3. [+] c1ccc(Nc2ncnc3cc(OCCOCn4ccnc4[N+](=O)[O-])c(OC)cc23)cc1 [EGFR-like scaffold]
 4. [-] CCN(CCCNc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)CC1
 5. [-] c1cc2c(Nc3ccc(Oc4ccccc4)cc3)ncnc2c1Cc1ccc(Cl)cc1
 6. [+] c1ccc(Nc2ncnc3cc4c(cc23)N(CCCN2CCOCC2)C(=O)CO4)cc1 [EGFR-like scaffold]
 7. [+] c1cc2c(Nc3cccc(Br)c3)ncnc2cn1 [EGFR-like scaffold]
 8. [+] CCN1CCN(Cc1ccc2ncnc(Nc3ccc(F)c(Cl)c3)c2c1)CC1 [EGFR-like scaffold]
 9. [-] c1ccc2c(c1)nc(O)n1ncnc(Nc3cccc(Br)c3)c1c21
10. [-] C(=O)NCc2cc3ncnc(Nc4ccc(F)c(Cl)c4)c3cc2NC1
11. [+] c1ccc(Nc2ncnc3cc(OCc4ccccn4)c5c(c23)N(CCCl)CC5)cc1 [EGFR-like scaffold]
12. [+] c1ccc(CNc2ccnc3oc4ccc(NC(=O)C5CCCCC5)cc4c23)cc1
13. [+] c1cc(CNc2n[nH]c3c(NCc4ccccc4)ncnc13)ccc2N1CCOCC1 [EGFR-like scaffold]
14. [+] CCCCCCNCc1cc(Cl)ccc1OCc1ccc2ncnc(Nc3ccc(OCc4cccc(F)c4)c(Cl)c3)c2c1 [EGFR-like scaffold

## 9. Cleanup checkpointów (limit 20GB)

In [16]:
import shutil

checkpoint_dir = f'{WORK_DIR}/training_checkpoints_stage2'
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
    print(f'Usunięto {checkpoint_dir}')

print('\nPozostało w /kaggle/working:')
!du -sh /kaggle/working/* 2>/dev/null


Pozostało w /kaggle/working:
892K	/kaggle/working/egfr_raw.csv
130M	/kaggle/working/qwen25_3b_smiles_egfr_lora


---
## Po zakończeniu

1. **Save Version → Quick Save** (lub Save & Run All) — to wykona ten notebook
   od początku do końca w nowej sesji i zapisze `qwen25_3b_smiles_egfr_lora/` jako Output.
2. Sprawdź zakładkę **Output** w wersji notebooka po zakończeniu commitu — powinien tam być
   folder adaptera.
3. Pobierz folder `qwen25_3b_smiles_egfr_lora/` (~150MB) — to jest Twój EGFR-specialized adapter.